In [1]:
##The script should make rlon correction, create corner coordinate for NarClim2.0 rotated grid and regrid using conservative regridding

#!check if this script corrects for rlon as instructed by NarClim2.0 readme if there is any further issue 
#!/g/data/gh02/dev/environments/intel_general/20211104/bin/python3.9
from pathlib import Path
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import xesmf

import pandas as pd
import re
import glob
import warnings
import os


import dask #parralelle processing 
import subprocess
import dask.array as da
from memory_profiler import profile
from filelock import FileLock
from dask.diagnostics import ProgressBar

import re # 👈 For extracting year from filenames
   
 
from dask.distributed import Client
client = Client()  # Or pass cluster details
client

# warnings.filterwarnings("ignore", category=UserWarning) #mute warnings 

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 7,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45115,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35909,Total threads: 1
Dashboard: /proxy/42483/status,Memory: 4.57 GiB
Nanny: tcp://127.0.0.1:41819,


In [2]:


# ========== 1. File Finder ========== #
def find_input_files(var, gcm, scenario, member, model, run, domain, institute, year):
    base_path = "/scratch/dt6/xw9650/tmp/regridded/"
    path_mask = f"{var}_{domain}_{gcm}_{scenario}_{member}_{institute}_{model}_{run}_day_{year}*.nc"
    return sorted(glob.glob(os.path.join(base_path, path_mask)))

# ========== 2. VP Function ========== #
def compute_vp(tas, hurs):
    """
    Compute vp using tas (K) and hurs (%).
    Returns DataArray with vp values (hPa).
    """
    T_c = tas - 273.15
    e_s = 6.112 * np.exp((17.67 * T_c) / (T_c + 243.5))
    vp = hurs * e_s / 100.0
  

    return vp.assign_attrs({
        "long_name": "Water Vapour Pressure",
        "units": "hPa",
        "description": "Calculated from air temperature and relative humidity using the Magnus equation"
    }).rename("vp")

# ========== 3. Processing + Save Function ========== #
def process_and_save_vp(tas_files, hurs_files, outfile, chunk_time='auto'):
    """
    Compute water vapour pressure(VP) from tas and hurs file lists, save to NetCDF.
    """
    # Load with chunking for Dask efficiency
    ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
    ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})

    # Rechunk after loading to avoid Dask warning
    # ds_tas = ds_tas.chunk({'time': chunk_time})
    # ds_hurs = ds_hurs.chunk({'time': chunk_time})
    
    # Compute vp
    vp = compute_vp(ds_tas['tas'], ds_hurs['hurs'])

    # Create dataset and save with progress bar
    vp_ds = vp.to_dataset()

    # Drop 'crs' if it exists
    if 'crs' in vp_ds.coords:
        vp_ds = vp_ds.drop_vars('crs')
        
    print("Final variables:", list(vp_ds.data_vars))
    
    with ProgressBar():
        vp_ds.to_netcdf(outfile, compute=True)


In [7]:

# ========== 4.1 GCM/RCM Loop Setup ========== #

gcms = ['ACCESS-ESM1-5']
scenarios = ['historical','ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r6i1p1f1']
models = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
runs = ['v1-r1']
institutes = ['NSW-Government']

output_dir = "/scratch/dt6/xw9650/tmp/derived/"
os.makedirs(output_dir, exist_ok=True)


# ========== Main Loop ========== #
for gcm in gcms:
    for scenario in scenarios:
        for member in members:
            for model in models:
                for run in runs:
                    for domain in domains:
                        for institute in institutes:

                            print(f"\n🔍 Searching files for: {gcm} | {scenario} | {model}")
                            
                            # Find all tas files for this config
                            tas_files = find_input_files('tas', gcm, scenario, member, model, run, domain, institute, year="")  # Empty year to get all

                            if not tas_files:
                                print("⚠️ No tas files found — skipping.")
                                continue

                            for tas_file in tas_files:
                                # Extract year from filename
                                match = re.search(r'_day_(\d{4})', tas_file)
                                if not match:
                                    print(f"⚠️ Year not found in {tas_file} — skipping.")
                                    continue

                                year = match.group(1)

                                # Find matching hurs file
                                hurs_files = find_input_files('hurs', gcm, scenario, member, model, run, domain, institute, year)

                                if not hurs_files:
                                    print(f"⚠️ No hurs file found for year {year} — skipping.")
                                    continue

                                # Output filename
                                outfile = (
                                    f"{output_dir}vp_{domain}_{gcm}_{scenario}_{member}_"
                                    f"{institute}_{model}_{run}_day_{year}0101-{year}1231.nc"
                                )

                                try:
                                    process_and_save_vp([tas_file], hurs_files, outfile)
                                    print(f"✅ Saved: {outfile}")
                                except Exception as e:
                                    print(f"❌ Failed: {outfile}\nError: {e}")



🔍 Searching files for: ACCESS-ESM1-5 | historical | NARCliM2-0-WRF412R3
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


...

Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


In [3]:

# ========== 4.2 GCM/RCM Loop Setup ========== #

gcms = ['EC-Earth3-Veg']
scenarios = [ 'historical', 'ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f1']
models = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
runs = ['v1-r1']
institutes = ['NSW-Government']

output_dir = "/scratch/dt6/xw9650/tmp/derived/"
os.makedirs(output_dir, exist_ok=True)


# ========== Main Loop ========== #
for gcm in gcms:
    for scenario in scenarios:
        for member in members:
            for model in models:
                for run in runs:
                    for domain in domains:
                        for institute in institutes:

                            print(f"\n🔍 Searching files for: {gcm} | {scenario} | {model}")
                            
                            # Find all tas files for this config
                            tas_files = find_input_files('tas', gcm, scenario, member, model, run, domain, institute, year="")  # Empty year to get all

                            if not tas_files:
                                print("⚠️ No tas files found — skipping.")
                                continue

                            for tas_file in tas_files:
                                # Extract year from filename
                                match = re.search(r'_day_(\d{4})', tas_file)
                                if not match:
                                    print(f"⚠️ Year not found in {tas_file} — skipping.")
                                    continue

                                year = match.group(1)

                                # Find matching hurs file
                                hurs_files = find_input_files('hurs', gcm, scenario, member, model, run, domain, institute, year)

                                if not hurs_files:
                                    print(f"⚠️ No hurs file found for year {year} — skipping.")
                                    continue

                                # Output filename
                                outfile = (
                                    f"{output_dir}vp_{domain}_{gcm}_{scenario}_{member}_"
                                    f"{institute}_{model}_{run}_day_{year}0101-{year}1231.nc"
                                )

                                try:
                                    process_and_save_vp([tas_file], hurs_files, outfile)
                                    print(f"✅ Saved: {outfile}")
                                except Exception as e:
                                    print(f"❌ Failed: {outfile}\nError: {e}")




🔍 Searching files for: EC-Earth3-Veg | historical | NARCliM2-0-WRF412R3
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19510101-19511231.nc


/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19520101-19521231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19530101-19531231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19540101-19541231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19550101-19551231.nc


/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19560101-19561231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19570101-19571231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19580101-19581231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_historical_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R3_v1-r1_day_19590101-19591231.nc


...

Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20880101-20881231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20890101-20891231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20900101-20901231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20910101-20911231.nc


/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20920101-20921231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20930101-20931231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20940101-20941231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20950101-20951231.nc


/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177744170.gadi-pbs/ipykernel_1470674/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20960101-20961231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20970101-20971231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20980101-20981231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_20990101-20991231.nc
Final variables: ['vp']
✅ Saved: /scratch/dt6/xw9650/tmp/derived/vp_AUS-18_EC-Earth3-Veg_ssp370_r1i1p1f1_NSW-Government_NARCliM2-0-WRF412R5_v1-r1_day_21000101-21001231.nc


In [4]:

# ========== 4.3 GCM/RCM Loop Setup ========== #

gcms = ['MPI-ESM1-2-HR']
scenarios = ['historical', 'ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f1']
models = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
runs = ['v1-r1']
institutes = ['NSW-Government']

output_dir = "/scratch/dt6/xw9650/tmp/derived/"
os.makedirs(output_dir, exist_ok=True)

# ========== Main Loop ========== #
for gcm in gcms:
    for scenario in scenarios:
        for member in members:
            for model in models:
                for run in runs:
                    for domain in domains:
                        for institute in institutes:

                            print(f"\n🔍 Searching files for: {gcm} | {scenario} | {model}")
                            
                            # Find all tas files for this config
                            tas_files = find_input_files('tas', gcm, scenario, member, model, run, domain, institute, year="")  # Empty year to get all

                            if not tas_files:
                                print("⚠️ No tas files found — skipping.")
                                continue

                            for tas_file in tas_files:
                                # Extract year from filename
                                match = re.search(r'_day_(\d{4})', tas_file)
                                if not match:
                                    print(f"⚠️ Year not found in {tas_file} — skipping.")
                                    continue

                                year = match.group(1)

                                # Find matching hurs file
                                hurs_files = find_input_files('hurs', gcm, scenario, member, model, run, domain, institute, year)

                                if not hurs_files:
                                    print(f"⚠️ No hurs file found for year {year} — skipping.")
                                    continue

                                # Output filename
                                outfile = (
                                    f"{output_dir}vp_{domain}_{gcm}_{scenario}_{member}_"
                                    f"{institute}_{model}_{run}_day_{year}0101-{year}1231.nc"
                                )

                                try:
                                    process_and_save_vp([tas_file], hurs_files, outfile)
                                    # print(f"✅ Saved: {outfile}")
                                except Exception as e:
                                    print(f"❌ Failed: {outfile}\nError: {e}")





🔍 Searching files for: MPI-ESM1-2-HR | historical | NARCliM2-0-WRF412R3
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


...

Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:32: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_tas = xr.open_mfdataset(tas_files, combine='by_coords', decode_times=False,chunks={'time': 365})
/jobfs/177734009.gadi-pbs/ipykernel_1543744/1938262415.py:33: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds_hurs = xr.open_mfdataset(hurs_files, combine='by_coords', decode_times=False,chunks={'time': 365})


Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']


In [5]:

# ========== 4.4 GCM/RCM Loop Setup ========== #

gcms = ['NorESM2-MM']
scenarios = ['historical','ssp126','ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f1']
models = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
runs = ['v1-r1']
institutes = ['NSW-Government']

output_dir = "/scratch/dt6/xw9650/tmp/derived/"
os.makedirs(output_dir, exist_ok=True)


# ========== Main Loop ========== #
for gcm in gcms:
    for scenario in scenarios:
        for member in members:
            for model in models:
                for run in runs:
                    for domain in domains:
                        for institute in institutes:

                            print(f"\n🔍 Searching files for: {gcm} | {scenario} | {model}")
                            
                            # Find all tas files for this config
                            tas_files = find_input_files('tas', gcm, scenario, member, model, run, domain, institute, year="")  # Empty year to get all

                            if not tas_files:
                                print("⚠️ No tas files found — skipping.")
                                continue

                            for tas_file in tas_files:
                                # Extract year from filename
                                match = re.search(r'_day_(\d{4})', tas_file)
                                if not match:
                                    print(f"⚠️ Year not found in {tas_file} — skipping.")
                                    continue

                                year = match.group(1)

                                # Find matching hurs file
                                hurs_files = find_input_files('hurs', gcm, scenario, member, model, run, domain, institute, year)

                                if not hurs_files:
                                    print(f"⚠️ No hurs file found for year {year} — skipping.")
                                    continue

                                # Output filename
                                outfile = (
                                    f"{output_dir}vp_{domain}_{gcm}_{scenario}_{member}_"
                                    f"{institute}_{model}_{run}_day_{year}0101-{year}1231.nc"
                                )

                                try:
                                    process_and_save_vp([tas_file], hurs_files, outfile)
                                    # print(f"✅ Saved: {outfile}")
                                except Exception as e:
                                    print(f"❌ Failed: {outfile}\nError: {e}")




🔍 Searching files for: NorESM2-MM | historical | NARCliM2-0-WRF412R3
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: [

In [6]:

# ========== 4.5 GCM/RCM Loop Setup ========== #

gcms = ['UKESM1-0-LL']
scenarios = ['historical', 'ssp26', 'ssp245','ssp370']
domains = ['AUS-18']
members = ['r1i1p1f2']
models = ['NARCliM2-0-WRF412R3', 'NARCliM2-0-WRF412R5']
runs = ['v1-r1']
institutes = ['NSW-Government']

output_dir = "/scratch/dt6/xw9650/tmp/derived/"
os.makedirs(output_dir, exist_ok=True)

import re  # 👈 For extracting year from filenames

# ========== Main Loop ========== #
for gcm in gcms:
    for scenario in scenarios:
        for member in members:
            for model in models:
                for run in runs:
                    for domain in domains:
                        for institute in institutes:

                            print(f"\n🔍 Searching files for: {gcm} | {scenario} | {model}")
                            
                            # Find all tas files for this config
                            tas_files = find_input_files('tas', gcm, scenario, member, model, run, domain, institute, year="")  # Empty year to get all

                            if not tas_files:
                                print("⚠️ No tas files found — skipping.")
                                continue

                            for tas_file in tas_files:
                                # Extract year from filename
                                match = re.search(r'_day_(\d{4})', tas_file)
                                if not match:
                                    print(f"⚠️ Year not found in {tas_file} — skipping.")
                                    continue

                                year = match.group(1)

                                # Find matching hurs file
                                hurs_files = find_input_files('hurs', gcm, scenario, member, model, run, domain, institute, year)

                                if not hurs_files:
                                    print(f"⚠️ No hurs file found for year {year} — skipping.")
                                    continue

                                # Output filename
                                outfile = (
                                    f"{output_dir}vp_{domain}_{gcm}_{scenario}_{member}_"
                                    f"{institute}_{model}_{run}_day_{year}0101-{year}1231.nc"
                                )

                                try:
                                    process_and_save_vp([tas_file], hurs_files, outfile)
                                    # print(f"✅ Saved: {outfile}")
                                except Exception as e:
                                    print(f"❌ Failed: {outfile}\nError: {e}")





🔍 Searching files for: UKESM1-0-LL | historical | NARCliM2-0-WRF412R3
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: ['vp']
Final variables: 